In [8]:
!pip install unsloth datasets huggingface-hub trl

In [9]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from datasets import load_dataset

In [10]:
# 1. Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load Llama 3.2 3B Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [11]:
# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [12]:
# 3. Load and Format Bitext Dataset
# We use the Bitext customer support dataset
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split = "train")

def format_bitext_prompts(examples):
    instructions = examples["instruction"]
    responses    = examples["response"]
    texts = []
    for instruction, response in zip(instructions, responses):
        # Llama 3.2 strict prompt format
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(format_bitext_prompts, batched = True)

In [13]:
FastLanguageModel.for_inference(model)  # enable faster inference mode

test_questions = [
    "I need to cancel my order, how can I do that?",
    "I want to change the delivery address for my purchase.",
    "I haven't received my refund yet, can you help?",
    "How do I track my package?",
]

print("=" * 60)
print("BEFORE TRAINING — Base Model Responses")
print("=" * 60)

for question in test_questions:
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {question}")
    print(f"A: {response.strip()}")
    print("-" * 60)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEFORE TRAINING — Base Model Responses


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: I need to cancel my order, how can I do that?
A: I'm happy to help you with canceling your order! However, I need a bit more information from you to assist you with that. Can you please provide the following details:

1. **Order number**: If you have a specific order number, please provide it to me. This will help me locate your order in our system.
2. **Order type**: Is it a subscription, one-time purchase, or a recurring order (e.g., membership, membership renewal)?
3. **Reason for cancellation**: What is the reason for canceling your order? Is it due to dissatisfaction with the product, change of mind, or something else?
4. **How did you place the order?**: Did you place the order through our website, mobile
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: I want to change the delivery address for my purchase.
A: Please provide the order number or other relevant details about your purchase, and I'll do my best to assist you with updating your delivery address.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: I haven't received my refund yet, can you help?
A: I'd be happy to help you with your refund issue. Can you please provide me with some more details? This will help me assist you more effectively. Please provide me with the following information:

1. **Your order number** (if you have it)
2. **The store or platform** where you made the purchase
3. **The reason for your refund request** (e.g. issue with the product, change of mind, etc.)
4. **Any relevant communication** you've had with the store or platform regarding your refund

The more information you provide, the better I'll be able to help you resolve your refund issue.
------------------------------------------------------------

Q: How do I track my package?
A: Tracking your package can be a bit tricky, but I'm here to guide you through the process. Here are the steps to track your package:

**If you haven't received your package yet:**

1. **Check your email**: Look for an email from the shipping carrier (e.g., UPS, FedEx, 

In [14]:
sft_config = SFTConfig(
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    gradient_accumulation_steps=4,
    per_device_train_batch_size=2,
    auto_find_batch_size=True,
    max_steps=100,
    logging_steps=1,
    optim="adamw_8bit",
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=max_seq_length,
    dataset_text_field="text",     # ← your dataset already has a "text" column
    dataset_num_proc=2,
    output_dir="outputs",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,    # ← processing_class, not tokenizer=
    args=sft_config,
    train_dataset=dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,872 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.105764
2,1.870322
3,2.022543
4,2.168329
5,2.114319
6,2.162551
7,2.069366
8,2.279024
9,2.030681
10,1.735619


TrainOutput(global_step=100, training_loss=1.2177734911441802, metrics={'train_runtime': 291.7911, 'train_samples_per_second': 2.742, 'train_steps_per_second': 0.343, 'total_flos': 2477523177222144.0, 'train_loss': 1.2177734911441802, 'epoch': 0.02977076510866329})

In [15]:
# Post-training inference — responses after fine-tuning on Bitext
FastLanguageModel.for_inference(model)  # re-enable inference mode after training

print("=" * 60)
print("AFTER TRAINING — Fine-tuned Model Responses")
print("=" * 60)

for question in test_questions:
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {question}")
    print(f"A: {response.strip()}")
    print("-" * 60)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER TRAINING — Fine-tuned Model Responses


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: I need to cancel my order, how can I do that?
A: Thank you for reaching out to us! I understand that you need assistance in canceling your order. To cancel your order, you can visit our website and log in to your account. Once you're logged in, you will be able to find the "Order History" or "Order Management" section, where you can locate the order you wish to cancel. From there, you should be able to initiate the cancellation process. If you encounter any difficulties or need further guidance, please don't hesitate to let me know. I'm here to assist you every step of the way.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: I want to change the delivery address for my purchase.
A: Sure! I'm here to help you with changing your delivery address for your purchase. To make the process smooth and seamless, could you please provide me with the details of your current delivery address and the new address you would like to update it to? Once I have this information, I will assist you in making the necessary changes and ensure that your purchase reaches its destination promptly and efficiently.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: I haven't received my refund yet, can you help?
A: I'm sorry to hear that you haven't received your refund yet. We understand the importance of timely refunds, and we are committed to ensuring that you receive your refund as soon as possible. To assist you further, could you please provide me with the order number or any other relevant details? This will help us locate your refund and expedite the process. Rest assured, we will do our best to resolve this issue promptly.
------------------------------------------------------------

Q: How do I track my package?
A: I'm on it! I understand your curiosity about tracking your package. To ensure you stay updated on the status of your shipment, you can follow these steps:

1. Visit the {{Website URL}}: Head over to our website at {{Website URL}}.
2. Access Your Account: Log in to your account using your credentials.
3. Track Your Order: Once you're logged in, navigate to the "{{Order History}}" or "{{Tracking Information}}" section.
4. E